In [ ]:
#Installs
#!pip install torch
#!pip install transformers

In [2]:
#Imports
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [14]:
#Loading a Pre-trained Emotion Detection Model
model_name = "j-hartmann/emotion-english-distilroberta-base"

#Load the Tokenizer which converts text into numbers which the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

#Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

#Define the list of emotions the model predicts
#For now, these emotions will be anger, fear, joy, neutral, and sadness but we may add more in the future
emotions = ["anger", "fear", "joy", "neutral", "sadness", "confusion"]

#Define the Minimum Confidence to Confidently Interpret an Emotion
#Set at .6 now but this is subject to change
confidence_threshold = .6

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# A function that recognizes the emotions behind a text
# The goal of this function is to try to recognize one of our listed emotions within a sentence
def emotion_prediction(sentence):

    # Convert the text input into a tensor PyTorch will use
    inputs = tokenizer(sentence, return_tensors="pt")

    # Get the model outputs which gives a raw score on each emotion
    outputs = model(**inputs)

    # Convert the raw score on each emotion into a probability
    probabilities = F.softmax(outputs.logits, dim=1)

    # Take the first row of probabilities due to an error being thrown when attempting
    probabilities_np_full = probabilities.detach().numpy()[0]

    # Keep only the probabilities corresponding to our 5 selected emotions
    # Model outputs order: ['anger','disgust','fear','joy','neutral','sadness','surprise']
    # We take indices: 0 (anger), 2 (fear), 3 (joy), 4 (neutral), 5 (sadness)
    probabilities_np = [
        probabilities_np_full[0],  # anger
        probabilities_np_full[2],  # fear
        probabilities_np_full[3],  # joy
        probabilities_np_full[4],  # neutral
        probabilities_np_full[5],  # sadness
        probabilities_np_full[6]]   # confusion

    # Takes the emotion with the highest probability score
    top_index = torch.argmax(torch.tensor(probabilities_np))
    detected_emotion = emotions[top_index]

    # Apply confidence threshold
    confidence = float(probabilities_np[top_index])
    meets_threshold = confidence >= confidence_threshold

    # Override emotion if below threshold
    if not meets_threshold:
        detected_emotion = "uncertain"

    # Return the detected emotion and confidence
    return {
    "Emotion": detected_emotion,
    "Confidence": confidence
    }

In [22]:
#Run the function on the test sentence 1
test_sentence1 = "I am very upset with your job and I will not be returning!"
result = emotion_prediction(test_sentence1)
print(result)

{'Emotion': 'anger', 'Confidence': 0.9542818665504456}


In [24]:
#Run the function on the test sentence 2
test_sentence2 = "I am afraid the procedure did not work as planned."
result2 = emotion_prediction(test_sentence2)
print(result2)

{'Emotion': 'fear', 'Confidence': 0.9893074631690979}


In [25]:
#Run the function on the test sentence 3
test_sentence3 = "Thank you so much for the successful surgery!"
result3 = emotion_prediction(test_sentence3)
print(result3)

{'Emotion': 'joy', 'Confidence': 0.920363187789917}


In [26]:
#Run the function on the test sentence 4
test_sentence4 = "Yeah I am doing alright."
result4 = emotion_prediction(test_sentence4)
print(result4)

{'Emotion': 'neutral', 'Confidence': 0.6603806614875793}


In [27]:
#Run the function on the test sentence 5
test_sentence5 = "Unfortunately, I have not been doing well since the procedure."
result5 = emotion_prediction(test_sentence5)
print(result5)

{'Emotion': 'sadness', 'Confidence': 0.928942084312439}


In [28]:
#Run the function on the test sentence 6
test_sentence6 = "Could you explain that again? I don't understand what you mean by that"
result6 = emotion_prediction(test_sentence6)
print(result6)

{'Emotion': 'confusion', 'Confidence': 0.9046690464019775}


In [40]:
# Defines the API function that takes a request dictionary
# and returns the detected emotion and confidence

def emotion_api(request):
    # Extract the text from the request
    text = request.get("text", "")

    # Call the emotion prediction function
    result = emotion_prediction(text)

    # Return the detected emotion and confidence using the correct keys
    return {
        "Emotion": result["Emotion"],
        "Confidence": result["Confidence"],
    }

In [41]:
# Test the API function by simulating a chatbot request

# Simulate a chatbot sending a request
test_request = {
    "text": "I am really frustrated with everything right now."
}

# Call the API function
response = emotion_api(test_request)

# Print the structured response
print(response)

{'Emotion': 'anger', 'Confidence': 0.9455840587615967}


In [45]:
#Temporary function that chooses a response based on the detected emotion
#I had ChatGPT generate temporary responses until we get a dynamic chatbot

def generate_chatbot_response(user_text):

    # Call the emotion API to detect emotion
    result = emotion_api({"text": user_text})

    # Get the detected emotion and confidence
    emotion = result["Emotion"]
    confidence = result["Confidence"]

    # Decide the chatbot's response based on the emotion
    if emotion == "joy":
        response = "That's great to hear!"
    elif emotion == "sadness":
        response = "I'm sorry you're feeling that way. Do you want to talk about it?"
    elif emotion == "anger":
        response = "That sounds frustrating. What happened?"
    elif emotion == "fear":
        response = "I understand that can be scary. Do you want to share more?"
    elif emotion == "neutral":
        response = "Thanks for sharing that with me."
    elif emotion == "confusion":
        response = "It sounds like things are unclear. Can you explain more?"
    elif emotion == "uncertain":
        response = "I’m not sure how to interpret that. Can you tell me more?"
    else:
        response = "Thanks for sharing!"

    # Return the chatbot response along with detected emotion and confidence
    return {
        "ChatbotResponse": response,
        "DetectedEmotion": emotion,
        "Confidence": confidence
    }

In [46]:
#Test the chatbot response function
user_input = "I am very upset with your job and I will not be returning!"
chatbot_output = generate_chatbot_response(user_input)
print(chatbot_output)

{'ChatbotResponse': 'That sounds frustrating. What happened?', 'DetectedEmotion': 'anger', 'Confidence': 0.9542818665504456}
